# Model training notebook

In [1]:
import torch
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler

import numpy as np
import math

from pathlib import Path
import time
import copy
import os
import random
from collections import Counter

In [2]:
torch.__version__

'2.10.0+cu128'

In [3]:
input_path_valtest = Path("../data/dataset_processed_split")
input_path_train = Path("../data/dataset_processed_split_synth")

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# Set random seeds for reproducibility

seed = 64

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


### Data preprocessing and loading

In [ ]:
# Normalization values are standard for ImageNet pre-trained models
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_tfms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),              # processed images are grayscale; convert to 3ch
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    normalize,

    # Helps regularize small/noisy datasets
    transforms.RandomErasing(p=0.10, scale=(0.02, 0.10)),
])

val_tfms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    normalize
])

image_datasets = {
    "train": datasets.ImageFolder(input_path_train / "train", transform=train_tfms),
    "val":   datasets.ImageFolder(input_path_valtest / "val",   transform=val_tfms),
    "test":  datasets.ImageFolder(input_path_valtest / "test",  transform=val_tfms),
}

num_classes = len(image_datasets["train"].classes)
dataset_size = {k: len(image_datasets[k]) for k in ["train", "val"]}

train_targets = [y for _, y in image_datasets["train"].samples]
counts = Counter(train_targets)

dataloaders = {
    "train": DataLoader(image_datasets["train"], batch_size=32, shuffle=True, num_workers=4, pin_memory=True),
    "val":   DataLoader(image_datasets["val"],   batch_size=32, shuffle=False, num_workers=4, pin_memory=True),
    "test":  DataLoader(image_datasets["test"],  batch_size=32, shuffle=False, num_workers=4, pin_memory=True),
}

print("Classes:", image_datasets["train"].classes)
print("Train class counts:", dict(counts))


Classes: ['angry', 'disgusted', 'fearful', 'happy', 'neutral', 'sad', 'surprised']
Train class counts: {0: 3736, 1: 3619, 2: 3743, 3: 5413, 4: 3784, 5: 3779, 6: 3744}


In [9]:
def one_hot(labels, num_classes, device):
    return F.one_hot(labels, num_classes=num_classes).float().to(device)

def rand_bbox(W, H, lam: float):
    cut_rat = math.sqrt(1.0 - lam)
    cut_w = max(int(W * cut_rat), 1)
    cut_h = max(int(H * cut_rat), 1)

    cx = torch.randint(0, W, (1,)).item()
    cy = torch.randint(0, H, (1,)).item()

    x1 = max(cx - cut_w // 2, 0)
    y1 = max(cy - cut_h // 2, 0)
    x2 = min(cx + cut_w // 2, W)
    y2 = min(cy + cut_h // 2, H)
    return x1, y1, x2, y2

def apply_mixup_cutmix(x, y, num_classes, mixup_alpha=0.4, cutmix_alpha=1.0,
                       p_mixup=0.0, p_cutmix=0.0):
    assert 0.0 <= p_mixup <= 1.0
    assert 0.0 <= p_cutmix <= 1.0
    assert p_mixup + p_cutmix <= 1.0

    device = x.device
    r = torch.rand(1, device=device).item()
    y1 = one_hot(y, num_classes, device)

    if r < p_mixup:
        lam = np.random.beta(mixup_alpha, mixup_alpha)
        idx = torch.randperm(x.size(0), device=device)
        x2, y2 = x[idx], y1[idx]
        x = x * lam + x2 * (1 - lam)
        y_soft = y1 * lam + y2 * (1 - lam)
        return x, y_soft

    if r < p_mixup + p_cutmix:
        lam = np.random.beta(cutmix_alpha, cutmix_alpha)
        idx = torch.randperm(x.size(0), device=device)
        x2, y2 = x[idx], y1[idx]

        _, _, H, W = x.shape
        x1b, y1b, x2b, y2b = rand_bbox(W, H, lam)
        x[:, :, y1b:y2b, x1b:x2b] = x2[:, :, y1b:y2b, x1b:x2b]

        box_area = (x2b - x1b) * (y2b - y1b)
        lam_adj = 1.0 - box_area / (W * H)
        y_soft = y1 * lam_adj + y2 * (1 - lam_adj)
        return x, y_soft

    return x, y1


In [10]:
class SoftTargetCrossEntropy(nn.Module):
    def __init__(self, weight=None, label_smoothing=0.0):
        super().__init__()
        if weight is not None and not torch.is_tensor(weight):
            weight = torch.tensor(weight, dtype=torch.float32)
        self.register_buffer("weight", weight)
        self.label_smoothing = float(label_smoothing)

    def forward(self, logits, target):
        log_probs = F.log_softmax(logits, dim=1)

        # allow hard targets optionally
        if target.dim() == 1:
            target = F.one_hot(target, num_classes=logits.size(1)).float().to(logits.device)

        # normalize (robustness)
        target = target / target.sum(dim=1, keepdim=True).clamp_min(1e-12)

        if self.label_smoothing > 0:
            C = target.size(1)
            target = target * (1 - self.label_smoothing) + self.label_smoothing / C

        loss = -(target * log_probs)
        if self.weight is not None:
            w = self.weight.to(dtype=logits.dtype, device=logits.device)
            loss = loss * w.unsqueeze(0)

        return loss.sum(dim=1).mean()


In [11]:
# class weights (optional; works nicely with soft-target loss too)
class_weights = torch.tensor([1.0 / counts[i] for i in range(num_classes)], dtype=torch.float)
class_weights = class_weights / class_weights.sum() * num_classes
class_weights = class_weights.to(device)


### Model training configurations

In [16]:
def train_model(model, dataloaders, device,
                criterion, optimizer, num_epochs=25, scheduler=None,
                early_stopping=True, patience=7, min_delta=1e-4):

    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    epochs_no_improve = 0

    # derive sizes from dataloaders to avoid mismatch bugs
    dataset_size = {k: len(dataloaders[k].dataset) for k in ["train", "val"]}

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    use_amp = (device.type == "cuda")
    scaler = GradScaler(enabled=use_amp)

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        print("-" * 10)

        for phase in ["train", "val"]:
            is_train = (phase == "train")
            model.train() if is_train else model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                with torch.set_grad_enabled(is_train):
                    with autocast(enabled=use_amp):
                        outputs = model(inputs)
                        loss = criterion(outputs, labels)

                    preds = outputs.argmax(dim=1)

                    if is_train:
                        scaler.scale(loss).backward()
                        scaler.step(optimizer)
                        scaler.update()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += (preds == labels).sum().item()

            epoch_loss = running_loss / dataset_size[phase]
            epoch_acc = running_corrects / dataset_size[phase]

            print(f"{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

            history[f"{phase}_loss"].append(epoch_loss)
            history[f"{phase}_acc"].append(epoch_acc)

            if phase == "val":
                if scheduler is not None:
                    if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                        scheduler.step(epoch_loss)
                    else:
                        scheduler.step()

                if epoch_acc > best_acc + min_delta:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1

        print(f"Current LR: {optimizer.param_groups[0]['lr']:.6f}")

        if early_stopping and epochs_no_improve >= patience:
            print(f"\nEarly stopping triggered after {epoch+1} epochs.")
            break

        print()

    time_elapsed = time.time() - since
    print(f"Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
    print(f"Best val Acc: {best_acc:.4f}")

    model.load_state_dict(best_model_wts)
    return model, history

In [18]:
label_smoothing = 0.05
criterion = SoftTargetCrossEntropy(label_smoothing=label_smoothing).to(device)


def set_trainable_convnext(model, train_backbone: bool):
    for p in model.parameters():
        p.requires_grad = train_backbone
    for p in model.classifier.parameters():
        p.requires_grad = True

model_ft = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)

num_ftrs = model_ft.classifier[2].in_features
model_ft.classifier[2] = nn.Linear(num_ftrs, num_classes)

model_ft = model_ft.to(device)

set_trainable_convnext(model_ft, train_backbone=False)

optimizer_stage1 = optim.SGD(
    model_ft.classifier.parameters(),
    lr=1e-2,
    momentum=0.9,
    weight_decay=5e-4,
    nesterov=True
)

model_ft, history1 = train_model(
    model_ft, dataloaders, device,
    criterion=criterion,
    optimizer=optimizer_stage1,
    num_epochs=5,
    scheduler=None,
    early_stopping=False
)

Epoch 1/5
----------


/tmp/ipykernel_78567/3819279412.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=use_amp)
/tmp/ipykernel_78567/3819279412.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


train Loss: 1.4976 Acc: 0.4658
val Loss: 1.4728 Acc: 0.4649
Current LR: 0.010000

Epoch 2/5
----------
train Loss: 1.4076 Acc: 0.5061
val Loss: 1.4749 Acc: 0.4494
Current LR: 0.010000

Epoch 3/5
----------
train Loss: 1.3813 Acc: 0.5216
val Loss: 1.3776 Acc: 0.5123
Current LR: 0.010000

Epoch 4/5
----------
train Loss: 1.3582 Acc: 0.5310
val Loss: 1.3685 Acc: 0.5204
Current LR: 0.010000

Epoch 5/5
----------
train Loss: 1.3510 Acc: 0.5321
val Loss: 1.3723 Acc: 0.5181
Current LR: 0.010000

Training complete in 3m 48s
Best val Acc: 0.5204


In [20]:
# Unfreeze backbone for fine-tuning
set_trainable_convnext(model_ft, train_backbone=True)

# Param groups: higher LR for classifier, lower LR for backbone
backbone_params = [p for n, p in model_ft.named_parameters() if not n.startswith("classifier.")]
classifier_params = list(model_ft.classifier.parameters())

optimizer_stage2 = optim.SGD(
    [
        {"params": classifier_params, "lr": 1e-3},
        {"params": backbone_params,   "lr": 1e-4},
    ],
    momentum=0.9,
    weight_decay=1e-4,
    nesterov=True
)

# ReduceLROnPlateau expects a metric (val loss) each epoch
scheduler_stage2 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_stage2,
    mode="min",
    factor=0.5,
    patience=5,
)


model_ft, history2 = train_model(
    model_ft, dataloaders, device,
    criterion=criterion,
    optimizer=optimizer_stage2,
    num_epochs=50,
    scheduler=scheduler_stage2,
    early_stopping=True,
    patience=10,
    min_delta=1e-4
)


Epoch 1/50
----------


/tmp/ipykernel_78567/3819279412.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=use_amp)
/tmp/ipykernel_78567/3819279412.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


train Loss: 1.2781 Acc: 0.5612
val Loss: 1.2600 Acc: 0.5667
Current LR: 0.001000

Epoch 2/50
----------
train Loss: 1.2120 Acc: 0.5893
val Loss: 1.2578 Acc: 0.5586
Current LR: 0.001000

Epoch 3/50
----------
train Loss: 1.1805 Acc: 0.6043
val Loss: 1.2332 Acc: 0.5799
Current LR: 0.001000

Epoch 4/50
----------
train Loss: 1.1580 Acc: 0.6153
val Loss: 1.2102 Acc: 0.5881
Current LR: 0.001000

Epoch 5/50
----------
train Loss: 1.1408 Acc: 0.6222
val Loss: 1.1868 Acc: 0.5958
Current LR: 0.001000

Epoch 6/50
----------
train Loss: 1.1201 Acc: 0.6332
val Loss: 1.1923 Acc: 0.5965
Current LR: 0.001000

Epoch 7/50
----------
train Loss: 1.1028 Acc: 0.6403
val Loss: 1.1725 Acc: 0.6067
Current LR: 0.001000

Epoch 8/50
----------
train Loss: 1.0956 Acc: 0.6421
val Loss: 1.1624 Acc: 0.6107
Current LR: 0.001000

Epoch 9/50
----------
train Loss: 1.0847 Acc: 0.6496
val Loss: 1.1485 Acc: 0.6178
Current LR: 0.001000

Epoch 10/50
----------
train Loss: 1.0729 Acc: 0.6497
val Loss: 1.1921 Acc: 0.6063
Cur

In [15]:
torch.save(model_ft.state_dict(), "../convnext_tiny.pth")
del model_ft
del optimizer_stage1
del optimizer_stage2
torch.cuda.empty_cache()